# 01. GBM 경로 시뮬레이션

기하 브라운 운동(Geometric Brownian Motion) 가정 하에서 기초자산 가격 경로를 생성하고 시각화한다.

$$dS_t = \mu S_t \, dt + \sigma S_t \, dW_t$$

이산화 형태:
$$S_{t+\Delta t} = S_t \exp\!\left[\left(r - \tfrac{1}{2}\sigma^2\right)\Delta t + \sigma\sqrt{\Delta t}\, Z_t\right], \quad Z_t \sim \mathcal{N}(0,1)$$

**분석 목표**
1. 샘플 경로 시각화
2. 만기 가격 분포 확인 (로그정규 분포 검증)
3. 변동성 수준별 경로 분산 비교

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import lognorm

from src import MarketAssumption, ContractSpec, SimulationSpec, simulate_paths, time_grid
from src.config import CHARTS_DIR, TABLES_DIR

CHARTS_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 130, 'font.size': 11})

## 1. 기본 파라미터 설정

In [ ]:
market = MarketAssumption(spot=100.0, rate=0.03, volatility=0.20)
contract = ContractSpec(strike=100.0, maturity=1.0, option_type='call')
sim = SimulationSpec(n_paths=200, n_steps=252, seed=42)

paths = simulate_paths(market, contract, sim)   # (200, 253)
t = time_grid(contract, sim)                    # (253,)

print(f'경로 배열 shape : {paths.shape}')
print(f'시간 격자 shape : {t.shape}')
print(f'S0 확인         : {paths[:, 0].mean():.2f}')

## 2. 샘플 경로 시각화

In [ ]:
N_DISPLAY = 50

fig, ax = plt.subplots(figsize=(10, 5))
for i in range(N_DISPLAY):
    ax.plot(t, paths[i], lw=0.6, alpha=0.5, color='steelblue')

# 평균 경로
mean_path = paths.mean(axis=0)
ax.plot(t, mean_path, lw=2, color='crimson', label='평균 경로')
ax.axhline(market.spot, lw=1, ls='--', color='gray', label=f'S0 = {market.spot}')

ax.set_title(f'GBM 샘플 경로 (σ={market.volatility}, r={market.rate}, T={contract.maturity}년)')
ax.set_xlabel('시간 (년)')
ax.set_ylabel('기초자산 가격')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'gbm_paths_sample.png', dpi=150)
plt.show()
print('저장: outputs/charts/gbm_paths_sample.png')

## 3. 만기 가격 분포 (로그정규 검증)

In [ ]:
# 대규모 시뮬레이션으로 분포 확인
sim_large = SimulationSpec(n_paths=50_000, n_steps=252, seed=42)
paths_large = simulate_paths(market, contract, sim_large)
terminal = paths_large[:, -1]

# 이론적 로그정규 파라미터
mu_ln = np.log(market.spot) + (market.rate - 0.5 * market.volatility**2) * contract.maturity
sigma_ln = market.volatility * np.sqrt(contract.maturity)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# 히스토그램 + 이론 분포
ax = axes[0]
x = np.linspace(terminal.min(), terminal.max(), 300)
pdf_theory = lognorm.pdf(x, s=sigma_ln, scale=np.exp(mu_ln))
ax.hist(terminal, bins=80, density=True, alpha=0.6, color='steelblue', label='MC 시뮬레이션')
ax.plot(x, pdf_theory, lw=2, color='crimson', label='이론 로그정규')
ax.set_title('만기 가격 분포')
ax.set_xlabel('$S_T$')
ax.set_ylabel('밀도')
ax.legend()
ax.grid(alpha=0.3)

# 로그 변환 후 정규분포 확인
ax2 = axes[1]
log_terminal = np.log(terminal)
x2 = np.linspace(log_terminal.min(), log_terminal.max(), 300)
from scipy.stats import norm
pdf_norm = norm.pdf(x2, loc=mu_ln, scale=sigma_ln)
ax2.hist(log_terminal, bins=80, density=True, alpha=0.6, color='steelblue', label='log($S_T$) MC')
ax2.plot(x2, pdf_norm, lw=2, color='crimson', label='이론 정규분포')
ax2.set_title('log($S_T$) 분포')
ax2.set_xlabel('log($S_T$)')
ax2.set_ylabel('밀도')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(CHARTS_DIR / 'terminal_price_distribution.png', dpi=150)
plt.show()
print('저장: outputs/charts/terminal_price_distribution.png')

## 4. 변동성 수준별 경로 분산 비교

In [ ]:
sigmas = [0.10, 0.20, 0.35, 0.50]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#F44336']
sim_cmp = SimulationSpec(n_paths=30, n_steps=252, seed=42)

fig, axes = plt.subplots(1, len(sigmas), figsize=(14, 4), sharey=False)

for ax, sigma, color in zip(axes, sigmas, colors):
    m = MarketAssumption(spot=100.0, rate=0.03, volatility=sigma)
    p = simulate_paths(m, contract, sim_cmp)
    for i in range(len(p)):
        ax.plot(t, p[i], lw=0.7, alpha=0.5, color=color)
    ax.plot(t, p.mean(axis=0), lw=2, color='black')
    ax.axhline(100, lw=1, ls='--', color='gray')
    ax.set_title(f'σ = {sigma}')
    ax.set_xlabel('시간 (년)')
    ax.grid(alpha=0.3)

axes[0].set_ylabel('기초자산 가격')
fig.suptitle('변동성 수준별 GBM 경로 비교', fontsize=13)
plt.tight_layout()
plt.savefig(CHARTS_DIR / 'gbm_volatility_comparison.png', dpi=150)
plt.show()
print('저장: outputs/charts/gbm_volatility_comparison.png')

## 요약

| 항목 | 결과 |
|---|---|
| 경로 배열 shape | (n_paths, n_steps+1) |
| 만기 가격 분포 | 로그정규 분포와 일치 |
| 변동성 증가 효과 | 경로 분산 확대, 상하방 이탈 경로 증가 |

GBM 경로 생성이 이론과 일치함을 확인했다. 다음 노트북에서 이 경로를 기반으로 유럽형 옵션 가격을 산출하고 Black-Scholes 해석해와 비교한다.